# Philips Agent Workflow System
## A2A Communication + MCP Tool Integration + Dynamic Planning + LLM

This notebook runs the complete multi-agent system on Google Colab.  
**Run each cell in order (Shift+Enter).**

---

## Step 1: Upload Project Files

Run the cell below — it will show an **Upload** button.  
Upload the `philips_3.1.zip` file you created from your laptop.

In [ ]:
import os

# Option A: Upload ZIP from your laptop
from google.colab import files
print("Click the button below to upload philips_3.1.zip")
uploaded = files.upload()

# Unzip into /content/
!unzip -qo philips_3.1.zip -d /content/
print("\n✅ Files uploaded and extracted.")

In [ ]:
# Find the project root (handles nested zip structures)
import glob

candidates = glob.glob('/content/**/run_demo.py', recursive=True)
if candidates:
    PROJECT_ROOT = os.path.dirname(candidates[0])
else:
    # Fallback: assume direct extraction
    PROJECT_ROOT = '/content/philips_3.1'

os.chdir(PROJECT_ROOT)
print(f"✅ Project root: {PROJECT_ROOT}")
print(f"\nFiles found:")
for f in sorted(os.listdir('.')):
    print(f"  {'📁' if os.path.isdir(f) else '📄'} {f}")

## Step 2: Install Dependencies

Installs PyPDF2, openpyxl, and fpdf2 (all free, no API keys needed).

In [ ]:
!pip install -q -r requirements.txt
print("\n✅ All dependencies installed.")

## Step 3: Generate Sample Healthcare Data

Creates `patient_data.xlsx` (patient vitals, device utilization, clinical outcomes) and `report.pdf` (clinical imaging AI report).

In [ ]:
!python sample_data/create_samples.py

print("\n📁 Sample files:")
for f in os.listdir('sample_data'):
    size = os.path.getsize(os.path.join('sample_data', f))
    print(f"  📄 {f} ({size:,} bytes)")

## Step 4: System Architecture Overview

Displays how the agents, message bus, and tool registry connect.

In [ ]:
import sys
sys.path.insert(0, PROJECT_ROOT)

print("""
┌─────────────────────────────────────────────────────────────────┐
│              PHILIPS AGENT WORKFLOW SYSTEM                     │
│      A2A Communication  +  MCP Tools  +  LLM-Ready            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   ┌───────────┐      A2A Messages       ┌───────────────┐      │
│   │   USER    │ ──────────────────────►  │  ORCHESTRATOR │      │
│   └───────────┘                          └───────┬───────┘      │
│                                                  │              │
│                              ┌───────────────────┼──────┐       │
│                              │    MESSAGE BUS    │      │       │
│                              │    (A2A Router)   │      │       │
│                              └───┬───────┬───────┘──────┘       │
│                                  │       │       │              │
│                     ┌────────────┤       │       ├──────────┐   │
│                     ▼            ▼       ▼       ▼          │   │
│              ┌──────────┐ ┌────────┐ ┌────────┐ ┌─────────┐│   │
│              │ PLANNER  │ │ READER │ │ANALYZER│ │SUMMARIZE││   │
│              │  Agent   │ │ Agent  │ │ Agent  │ │  Agent  ││   │
│              └──────────┘ └───┬────┘ └───┬────┘ └────┬────┘│   │
│                               │          │           │      │   │
│                  ┌────────────┴──────────┴───────────┴──┐   │   │
│                  │       MCP TOOL REGISTRY              │   │   │
│                  │  read_pdf  │ read_excel │ read_text  │   │   │
│                  │  extract_q │ summarize  │ keywords   │   │   │
│                  │  analyze   │ llm_enhance│            │   │   │
│                  └──────────────────────────────────────┘   │   │
│                                                                 │
│   ┌─ LLM ADAPTER ───────────────────────────────────────┐      │
│   │  OpenAI / Ollama / Mock — pluggable AI backend      │      │
│   └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
""")

## Step 5: Explore the MCP Tool Registry

Shows all 8 registered tools with their metadata — agents discover these at runtime.

In [ ]:
from agent_workflow.main import build_tool_registry

registry = build_tool_registry()
tools = registry.list_tools()

print(f"📦 Registered MCP Tools ({len(tools)})")
print("=" * 65)
for tool in tools:
    print(f"\n  🔧 [{tool['name']}]")
    print(f"     Description: {tool['description']}")
    print(f"     Tags:        {', '.join(tool['tags'])}")
    print(f"     Params:      {tool['input_schema']}")

print("\n" + "=" * 65)
print("\n🔍 Dynamic Tool Search (agents use this at runtime):")
for query in ["pdf", "analysis", "summary", "llm"]:
    matches = registry.find_for_task([query])
    names = [m['name'] for m in matches]
    print(f"   Query '{query}' → {names}")

## Step 6: LLM Adapter Demo (Mock Mode)

Shows the pluggable LLM backend generating clinical insights — no API key needed.

In [ ]:
from agent_workflow.llm_adapter import LLMAdapter

adapter = LLMAdapter()  # auto-detects, falls back to mock
print(f"🤖 LLM Backend: {adapter.backend_name}")
print("=" * 65)

# Enhancement demo
sample_summary = (
    "AI-assisted imaging reduced diagnostic errors by 15%. "
    "Telemedicine adoption increased 23%. Remote monitoring improved outcomes."
)
print("\n📝 Input Summary:")
print(f"   {sample_summary}")

enhanced = adapter.enhance_summary(sample_summary, context="Healthcare AI review")
print(f"\n✨ LLM-Enhanced Output:")
print(f"   {enhanced}")

# Clinical insight demo
patient_stats = {
    "Heart_Rate_BPM": {"mean": 81.2, "min": 68, "max": 96},
    "SpO2_Percent": {"mean": 95.8, "min": 91, "max": 99},
    "Risk_Level": {"Critical": 3, "High": 2, "Medium": 3, "Low": 4},
}
print("\n📊 Patient Statistics:")
for k, v in patient_stats.items():
    print(f"   {k}: {v}")

insights = adapter.generate_insights(patient_stats)
print("\n💡 LLM-Generated Clinical Insights:")
for i, insight in enumerate(insights, 1):
    print(f"   {i}. {insight}")

## Step 7: Demo — Summarize Clinical Document & Extract Questions

Full A2A workflow: Orchestrator → Planner → Reader → Analyzer → Summarizer

In [ ]:
from agent_workflow.main import run_task

results = run_task(
    task="Summarize this document and extract all questions",
    files=["sample_data/sample.txt"],
)

## Step 8: Demo — Analyze Patient Data & Device Utilization

Reads Excel with 3 sheets of healthcare data, generates statistical analysis.

In [ ]:
results = run_task(
    task="Analyze data and provide insights and statistics",
    files=["sample_data/patient_data.xlsx"],
)

## Step 9: Demo — Clinical Keyword Extraction

Extracts the most important medical/clinical keywords from the document.

In [ ]:
results = run_task(
    task="Extract the most important keywords and topics",
    files=["sample_data/sample.txt"],
)

## Step 10: Demo — Multi-File Processing

Processes both text AND Excel files in a single task — planner adapts automatically.

In [ ]:
results = run_task(
    task="Summarize all documents and analyze data trends",
    files=["sample_data/sample.txt", "sample_data/patient_data.xlsx"],
)

## Step 11: Run Unit Tests

Runs all 24 tests to verify system integrity.

In [ ]:
!pip install -q pytest
!python -m pytest tests/ -v

## Step 12: Inspect A2A Message Flow (Under the Hood)

Shows how messages flow through the system.

In [ ]:
from agent_workflow.main import setup
from agent_workflow.core.models import Message, MessageType

orchestrator, trace, bus = setup()

# Manually send A2A messages to show the protocol
print("🔗 A2A Message Flow Demo")
print("=" * 65)

# 1. Orchestrator sends to Planner
msg1 = Message(
    sender="orchestrator",
    recipient="planner",
    msg_type=MessageType.REQUEST,
    payload={"task": "Summarize document", "files": ["report.txt"]},
)
bus.send(msg1)
print(f"\n📤 SENT: {msg1}")
print(f"   Payload: {msg1.payload}")

# 2. Planner receives from its queue
received = bus.receive("planner")
print(f"\n📥 RECEIVED by planner: {received}")
print(f"   Correlation ID: {received.correlation_id[:8]}...")

# 3. Show bus state
print(f"\n📊 Bus stats: {bus.message_count()} total messages logged")
print(f"   Planner queue empty: {not bus.has_messages('planner')}")

print("\n" + "=" * 65)
print("\n✅ This is exactly how agents communicate in the full workflow.")
print("   No direct function calls between agents — only Messages via the Bus.")

---
## ✅ All Demos Complete

### Key Points for Interview:
- **A2A**: Agents communicate only via Message objects through the MessageBus
- **MCP**: Tools are registered centrally; agents discover and invoke them dynamically
- **Non-Hardcoded**: PlannerAgent generates execution plans at runtime
- **Adaptive**: Failed steps are skipped; remaining steps continue
- **LLM-Ready**: Pluggable AI backend (OpenAI, Ollama, or offline mock)
- **Healthcare**: Patient vitals, device utilization, clinical outcomes, medical imaging
- **Tested**: 24 unit tests covering all components